# Campaign 25 manual split audit

This notebook renders the exact supervised multitile targets used by Campaign 25 for all 11 training fragments. Each fragment has its own visualization cell.

Color interpretation:

- **red**: training ink
- **yellow**: training papyrus ring negative
- **blue**: validation ink
- **green**: validation papyrus ring negative
- **cyan**: explicit manual negative
- **gray**: the real continuous ink label at 25% opacity

The helper uses the production `DataManager`, `_fetch_label_mt()`, and `_fetch_mask_mt()` paths. Each displayed pixel represents one 16 px model target. Fragments without a continuous `inklabels/<id>.png` fall back to the eroded label for the gray reference layer.

In [ ]:
from pathlib import Path
import gc

import cv2
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np

from campaign_archs_25 import TESTS, _SCROLL_IDS, build_config
from utils.dataloader import DataManager

ROOT = Path.cwd()
if ROOT.name == "old":
    ROOT = ROOT.parent

SAVE_PNG = True
OUT_DIR = ROOT / "output" / "campaign25_split_audit"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SCROLL_NAMES = {
    20260115000000: "PHerc0139 w044",
    20260317000000: "PHerc0139 w035",
    20250223000000: "PHerc0139 w059",
    20251111010954: "PHerc0172 w068",
    20251112000002: "PHerc0172 w087",
    20240304141531: "PHerc1667 w013",
    20240304144031: "PHerc1667 w018",
    20250919125754: "PHerc0009B 487",
    20231210121321: "PHercParis4",
    20250628074500: "PHerc0500P2",
    20260226000000: "PHerc0814",
}

assert tuple(SCROLL_NAMES) == _SCROLL_IDS

COLORS = {
    "train_ink": np.array([255, 0, 0], dtype=np.uint8),
    "train_papyrus": np.array([255, 220, 0], dtype=np.uint8),
    "valid_ink": np.array([0, 90, 255], dtype=np.uint8),
    "valid_papyrus": np.array([0, 190, 70], dtype=np.uint8),
    "explicit_negative": np.array([0, 255, 255], dtype=np.uint8),
}


def _read_gray(path):
    image = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if image is None:
        raise FileNotFoundError(path)
    return image


def _real_inklabel(scroll_id):
    candidates = (
        (ROOT / "inklabels" / f"{scroll_id}.png", "continuous inklabel"),
        (ROOT / "inklabels" / "2_4um" / f"{scroll_id}.png", "archived 2.4 µm inklabel"),
        (ROOT / "inklabels" / "1_1um" / f"{scroll_id}.png", "archived 1.1 µm inklabel"),
        (ROOT / "eroded_inklabels" / f"{scroll_id}.png", "eroded fallback"),
    )
    for path, description in candidates:
        if path.exists():
            return _read_gray(path), description
    raise FileNotFoundError(f"no ink reference found for {scroll_id}")


def _campaign25_audit_config():
    cfg = build_config(TESTS[0])
    cfg.data.preload_volumes = False
    cfg.data.mask_memmap = False
    cfg.data.mask_bitpack = False
    cfg.data.character_balanced_sampling = False
    cfg.tra.character_macro_metrics = False
    cfg.dl.data_aug = False
    return cfg


def _collect_targets(dataset, maps):
    n = dataset._mt_grid
    sub = dataset._mt_sub
    for _, y_off, x_off in dataset.block_coords:
        labels = dataset._fetch_label_mt(y_off, x_off).numpy()
        valid = dataset._fetch_mask_mt(y_off, x_off).numpy() > 0
        y0, _, x0, _ = dataset._mt_center_bounds(y_off, x_off)
        for index in np.flatnonzero(valid):
            iy, ix = divmod(int(index), n)
            ys = y0 + iy * sub
            xs = x0 + ix * sub
            cy = (ys + sub // 2) // sub
            cx = (xs + sub // 2) // sub
            if not (0 <= cy < maps["shape"][0] and 0 <= cx < maps["shape"][1]):
                continue
            value = labels[index]
            if value < 0:
                maps["explicit_negative"][cy, cx] = True
            elif value > 0:
                maps[f"{maps['split']}_ink"][cy, cx] = True
            else:
                maps[f"{maps['split']}_papyrus"][cy, cx] = True


def render_scroll(scroll_id, name):
    cfg = _campaign25_audit_config()
    scroll_index = list(_SCROLL_IDS).index(int(scroll_id))
    dm = DataManager(
        cfg,
        scroll_id=int(scroll_id),
        domain_id=scroll_index,
        character_namespace=scroll_index,
    )
    train_set, valid_set = dm.get_datasets()

    height, width = dm.labels.shape
    sub = int(cfg.model.multitile_subtile)
    display_shape = (int(np.ceil(height / sub)) + 1, int(np.ceil(width / sub)) + 1)
    target_maps = {key: np.zeros(display_shape, dtype=bool) for key in COLORS}
    target_maps["shape"] = display_shape

    target_maps["split"] = "train"
    _collect_targets(train_set, target_maps)
    target_maps["split"] = "valid"
    _collect_targets(valid_set, target_maps)
    target_maps.pop("split")
    target_maps.pop("shape")

    train_any = target_maps["train_ink"] | target_maps["train_papyrus"] | target_maps["explicit_negative"]
    valid_any = target_maps["valid_ink"] | target_maps["valid_papyrus"]
    assert not np.any(train_any & valid_any), f"train/validation overlap for {scroll_id}"

    raw, raw_source = _real_inklabel(scroll_id)
    background = cv2.resize(
        raw,
        (display_shape[1], display_shape[0]),
        interpolation=cv2.INTER_AREA,
    )
    rgb = np.repeat(background[..., None], 3, axis=2).astype(np.float32) * 0.25
    rgb = rgb.astype(np.uint8)

    for key in ("train_ink", "train_papyrus", "valid_ink", "valid_papyrus"):
        rgb[target_maps[key]] = COLORS[key]
    rgb[target_maps["explicit_negative"]] = COLORS["explicit_negative"]

    counts = {key: int(target_maps[key].sum()) for key in COLORS}
    supervised = sum(counts.values())
    fig, ax = plt.subplots(figsize=(18, max(6, 18 * height / width)))
    ax.imshow(rgb, interpolation="nearest")
    ax.set_title(
        f"{name} — {scroll_id}\n"
        f"real ink reference={raw_source} at 25% opacity; "
        f"supervised targets={supervised:,}; target={sub}px"
    )
    ax.axis("off")
    legend = [
        mpatches.Patch(color=COLORS["train_ink"] / 255, label=f"train ink ({counts['train_ink']:,})"),
        mpatches.Patch(color=COLORS["train_papyrus"] / 255, label=f"train papyrus ({counts['train_papyrus']:,})"),
        mpatches.Patch(color=COLORS["valid_ink"] / 255, label=f"validation ink ({counts['valid_ink']:,})"),
        mpatches.Patch(color=COLORS["valid_papyrus"] / 255, label=f"validation papyrus ({counts['valid_papyrus']:,})"),
        mpatches.Patch(color=COLORS["explicit_negative"] / 255, label=f"explicit negative ({counts['explicit_negative']:,})"),
        mpatches.Patch(color=(0.25, 0.25, 0.25), label=f"real ink label ({raw_source}, 25%)"),
    ]
    ax.legend(handles=legend, loc="upper right", fontsize=9, framealpha=0.95)
    plt.tight_layout()

    out_path = OUT_DIR / f"{scroll_id}_campaign25_split.png"
    if SAVE_PNG:
        fig.savefig(out_path, dpi=180, bbox_inches="tight")
        print(f"saved {out_path}")
    print(counts)
    plt.show()

    del train_set, valid_set, dm, raw, background, rgb, target_maps
    gc.collect()


print("Campaign 25 audit helper ready for 11 fragments")

In [ ]:
OVERLAY_ALPHA = 0.5





def _blend_target_color(rgb, mask, color):

    if not np.any(mask):

        return

    rgb[mask] = np.clip(

        (1.0 - OVERLAY_ALPHA) * rgb[mask].astype(np.float32)

        + OVERLAY_ALPHA * np.asarray(color, dtype=np.float32),

        0,

        255,

    ).astype(np.uint8)





def render_scroll(scroll_id, name):

    cfg = _campaign25_audit_config()

    scroll_index = list(_SCROLL_IDS).index(int(scroll_id))

    dm = DataManager(

        cfg,

        scroll_id=int(scroll_id),

        domain_id=scroll_index,

        character_namespace=scroll_index,

    )

    train_set, valid_set = dm.get_datasets()



    height, width = dm.labels.shape

    sub = int(cfg.model.multitile_subtile)

    display_shape = (int(np.ceil(height / sub)) + 1, int(np.ceil(width / sub)) + 1)

    target_maps = {key: np.zeros(display_shape, dtype=bool) for key in COLORS}

    target_maps["shape"] = display_shape

    target_maps["split"] = "train"

    _collect_targets(train_set, target_maps)

    target_maps["split"] = "valid"

    _collect_targets(valid_set, target_maps)

    target_maps.pop("split")

    target_maps.pop("shape")



    train_any = target_maps["train_ink"] | target_maps["train_papyrus"] | target_maps["explicit_negative"]

    valid_any = target_maps["valid_ink"] | target_maps["valid_papyrus"]

    assert not np.any(train_any & valid_any), f"train/validation overlap for {scroll_id}"



    raw, raw_source = _real_inklabel(scroll_id)

    background = cv2.resize(raw, (display_shape[1], display_shape[0]), interpolation=cv2.INTER_AREA)

    rgb = np.repeat(background[..., None], 3, axis=2).astype(np.uint8)

    for key in ("train_ink", "train_papyrus", "valid_ink", "valid_papyrus", "explicit_negative"):

        _blend_target_color(rgb, target_maps[key], COLORS[key])



    counts = {key: int(target_maps[key].sum()) for key in COLORS}

    supervised = sum(counts.values())

    fig, ax = plt.subplots(figsize=(18, max(6, 18 * height / width)))

    ax.imshow(rgb, interpolation="nearest")

    ax.set_title(

        f"{name} — {scroll_id}\n"

        f"50% supervision overlay over {raw_source}; "

        f"supervised targets={supervised:,}; target={sub}px"

    )

    ax.axis("off")

    legend = [

        mpatches.Patch(color=COLORS["train_ink"] / 255, alpha=OVERLAY_ALPHA, label=f"train ink ({counts['train_ink']:,})"),

        mpatches.Patch(color=COLORS["train_papyrus"] / 255, alpha=OVERLAY_ALPHA, label=f"train papyrus ({counts['train_papyrus']:,})"),

        mpatches.Patch(color=COLORS["valid_ink"] / 255, alpha=OVERLAY_ALPHA, label=f"validation ink ({counts['valid_ink']:,})"),

        mpatches.Patch(color=COLORS["valid_papyrus"] / 255, alpha=OVERLAY_ALPHA, label=f"validation papyrus ({counts['valid_papyrus']:,})"),

        mpatches.Patch(color=COLORS["explicit_negative"] / 255, alpha=OVERLAY_ALPHA, label=f"explicit negative ({counts['explicit_negative']:,})"),

        mpatches.Patch(color=(0.5, 0.5, 0.5), label=f"ink reference ({raw_source})"),

    ]

    ax.legend(handles=legend, loc="upper right", fontsize=9, framealpha=0.95)

    plt.tight_layout()



    out_path = OUT_DIR / f"{scroll_id}_campaign25_split.png"

    if SAVE_PNG:

        fig.savefig(out_path, dpi=180, bbox_inches="tight")

        print(f"saved {out_path}")

    print(counts)

    plt.show()



    del train_set, valid_set, dm, raw, background, rgb, target_maps

    gc.collect()





print("All split overlays use 50% alpha")

In [ ]:
render_scroll(20260115000000, SCROLL_NAMES[20260115000000])

In [ ]:
render_scroll(20260317000000, SCROLL_NAMES[20260317000000])

In [ ]:
render_scroll(20250223000000, SCROLL_NAMES[20250223000000])

In [ ]:
render_scroll(20251111010954, SCROLL_NAMES[20251111010954])

In [ ]:
render_scroll(20251112000002, SCROLL_NAMES[20251112000002])

In [ ]:
render_scroll(20240304141531, SCROLL_NAMES[20240304141531])

In [ ]:
render_scroll(20240304144031, SCROLL_NAMES[20240304144031])

In [ ]:
render_scroll(20250919125754, SCROLL_NAMES[20250919125754])

In [ ]:
render_scroll(20231210121321, SCROLL_NAMES[20231210121321])

In [ ]:
render_scroll(20250628074500, SCROLL_NAMES[20250628074500])

In [ ]:
render_scroll(20260226000000, SCROLL_NAMES[20260226000000])

## Configurable ring and `pos_only` audit



`ring_negatives=False` means supervision over the full papyrus mask; it does not disable negative supervision. `ring_shell_r=0` selects the automatically balanced ring width. With `multitile_pos_only=True`, mixed positive windows retain only positive targets, while ink-free ring windows still provide negatives.



The audit below reports unique target cells, repeated target occurrences across overlapping windows, mixed-sign windows, and direct positive-negative boundary edges.

In [ ]:
def _audit_config(*, ring_negatives=True, ring_source="closed", close_r=3, gap_r=3, shell_r=2, pos_only=True):

    cfg = _campaign25_audit_config()

    cfg.data.ring_negatives = bool(ring_negatives)

    cfg.data.ring_label_source = str(ring_source)

    cfg.data.ring_close_r = int(close_r)

    cfg.data.ring_gap_r = int(gap_r)

    cfg.data.ring_shell_r = int(shell_r)

    cfg.data.multitile_pos_only = bool(pos_only)

    return cfg





def _empty_audit_maps(shape):

    return {

        "positive": np.zeros(shape, dtype=bool),

        "negative": np.zeros(shape, dtype=bool),

        "explicit_negative": np.zeros(shape, dtype=bool),

        "positive_hits": np.zeros(shape, dtype=np.uint32),

        "negative_hits": np.zeros(shape, dtype=np.uint32),

    }





def _collect_scenario_targets(dataset, maps):

    n = dataset._mt_grid

    sub = dataset._mt_sub

    stats = {

        "windows": 0,

        "positive_windows": 0,

        "negative_only_windows": 0,

        "mixed_sign_windows": 0,

        "supervised_occurrences": 0,

        "positive_occurrences": 0,

        "negative_occurrences": 0,

    }

    for _, y_off, x_off in dataset.block_coords:

        labels = dataset._fetch_label_mt(y_off, x_off).numpy()

        valid = dataset._fetch_mask_mt(y_off, x_off).numpy() > 0

        positive = valid & (labels > 0)

        negative = valid & (labels <= 0)

        has_positive = bool(positive.any())

        has_negative = bool(negative.any())

        stats["windows"] += 1

        stats["positive_windows"] += int(has_positive)

        stats["negative_only_windows"] += int(has_negative and not has_positive)

        stats["mixed_sign_windows"] += int(has_positive and has_negative)

        stats["supervised_occurrences"] += int(valid.sum())

        stats["positive_occurrences"] += int(positive.sum())

        stats["negative_occurrences"] += int(negative.sum())



        y0, _, x0, _ = dataset._mt_center_bounds(y_off, x_off)

        for index in np.flatnonzero(valid):

            iy, ix = divmod(int(index), n)

            ys = y0 + iy * sub

            xs = x0 + ix * sub

            cy = (ys + sub // 2) // sub

            cx = (xs + sub // 2) // sub

            if not (0 <= cy < maps["positive"].shape[0] and 0 <= cx < maps["positive"].shape[1]):

                continue

            if labels[index] > 0:

                maps["positive"][cy, cx] = True

                maps["positive_hits"][cy, cx] += 1

            else:

                maps["negative"][cy, cx] = True

                maps["negative_hits"][cy, cx] += 1

                if labels[index] < 0:

                    maps["explicit_negative"][cy, cx] = True

    return stats





def _adjacent_boundary_edges(positive, negative):

    horizontal = (positive[:, :-1] & negative[:, 1:]) | (negative[:, :-1] & positive[:, 1:])

    vertical = (positive[:-1, :] & negative[1:, :]) | (negative[:-1, :] & positive[1:, :])

    return int(horizontal.sum() + vertical.sum())





def _scenario_stats(maps, window_stats):

    positive = maps["positive"]

    negative = maps["negative"]

    both = positive & negative

    supervised = positive | negative

    repeated = (maps["positive_hits"] + maps["negative_hits"]) > 1

    stats = dict(window_stats)

    stats.update({

        "unique_supervised_cells": int(supervised.sum()),

        "unique_positive_cells": int(positive.sum()),

        "unique_negative_cells": int(negative.sum()),

        "unique_explicit_negative_cells": int(maps["explicit_negative"].sum()),

        "cells_with_both_labels": int(both.sum()),

        "cells_seen_multiple_times": int(repeated.sum()),

        "positive_negative_boundary_edges": _adjacent_boundary_edges(positive, negative),

    })

    stats["positive_fraction"] = stats["unique_positive_cells"] / max(stats["unique_supervised_cells"], 1)

    stats["mixed_window_fraction"] = stats["mixed_sign_windows"] / max(stats["windows"], 1)

    return stats





def audit_ring_scenario(scroll_id, name, scenario, **options):

    cfg = _audit_config(**options)

    scroll_index = list(_SCROLL_IDS).index(int(scroll_id))

    dm = DataManager(

        cfg,

        scroll_id=int(scroll_id),

        domain_id=scroll_index,

        character_namespace=scroll_index,

    )

    train_set, valid_set = dm.get_datasets()

    height, width = dm.labels.shape

    sub = int(cfg.model.multitile_subtile)

    shape = (int(np.ceil(height / sub)) + 1, int(np.ceil(width / sub)) + 1)

    train_maps = _empty_audit_maps(shape)

    valid_maps = _empty_audit_maps(shape)

    train_stats = _scenario_stats(train_maps, _collect_scenario_targets(train_set, train_maps))

    valid_stats = _scenario_stats(valid_maps, _collect_scenario_targets(valid_set, valid_maps))



    raw, raw_source = _real_inklabel(scroll_id)

    background = cv2.resize(raw, (shape[1], shape[0]), interpolation=cv2.INTER_AREA)

    rgb = (np.repeat(background[..., None], 3, axis=2).astype(np.float32) * 0.25).astype(np.uint8)

    rgb[train_maps["negative"]] = COLORS["train_papyrus"]

    rgb[valid_maps["negative"]] = COLORS["valid_papyrus"]

    rgb[train_maps["positive"]] = COLORS["train_ink"]

    rgb[valid_maps["positive"]] = COLORS["valid_ink"]

    rgb[train_maps["explicit_negative"]] = COLORS["explicit_negative"]



    fig, ax = plt.subplots(figsize=(18, max(6, 18 * height / width)))

    ax.imshow(rgb, interpolation="nearest")

    ax.set_title(

        f"{name} — {scroll_id} — {scenario}\n"

        f"ring={cfg.data.ring_negatives}, source={cfg.data.ring_label_source}, "

        f"close={cfg.data.ring_close_r}, gap={cfg.data.ring_gap_r}, "

        f"shell={cfg.data.ring_shell_r}, pos_only={cfg.data.multitile_pos_only}"

    )

    ax.axis("off")

    ax.legend(handles=[

        mpatches.Patch(color=COLORS["train_ink"] / 255, label=f"train ink ({train_stats['unique_positive_cells']:,})"),

        mpatches.Patch(color=COLORS["train_papyrus"] / 255, label=f"train negative ({train_stats['unique_negative_cells']:,})"),

        mpatches.Patch(color=COLORS["valid_ink"] / 255, label=f"validation ink ({valid_stats['unique_positive_cells']:,})"),

        mpatches.Patch(color=COLORS["valid_papyrus"] / 255, label=f"validation negative ({valid_stats['unique_negative_cells']:,})"),

        mpatches.Patch(color=(0.25, 0.25, 0.25), label=f"ink reference ({raw_source}, 25%)"),

    ], loc="upper right", fontsize=9, framealpha=0.95)

    plt.tight_layout()



    if SAVE_PNG:

        safe_name = "".join(ch if ch.isalnum() or ch in "-_" else "_" for ch in scenario)

        out_path = OUT_DIR / f"{scroll_id}_{safe_name}.png"

        fig.savefig(out_path, dpi=180, bbox_inches="tight")

        print(f"saved {out_path}")

    print(f"{scenario} train: {train_stats}")

    print(f"{scenario} valid: {valid_stats}")

    plt.show()



    del train_set, valid_set, dm, raw, background, rgb, train_maps, valid_maps

    gc.collect()

    return {"scenario": scenario, "train": train_stats, "valid": valid_stats}





print("Configurable ring audit helper ready")

In [ ]:
def _collect_scenario_targets(dataset, maps):

    n = dataset._mt_grid

    sub = dataset._mt_sub

    stats = {

        "windows": 0,

        "positive_windows": 0,

        "negative_only_windows": 0,

        "mixed_sign_windows": 0,

        "mixed_regular_negative_windows": 0,

        "mixed_explicit_negative_windows": 0,

        "supervised_occurrences": 0,

        "positive_occurrences": 0,

        "regular_negative_occurrences": 0,

        "explicit_negative_occurrences": 0,

    }

    for _, y_off, x_off in dataset.block_coords:

        labels = dataset._fetch_label_mt(y_off, x_off).numpy()

        valid = dataset._fetch_mask_mt(y_off, x_off).numpy() > 0

        positive = valid & (labels > 0)

        regular_negative = valid & (labels == 0)

        explicit_negative = valid & (labels < 0)

        negative = regular_negative | explicit_negative

        has_positive = bool(positive.any())

        has_negative = bool(negative.any())

        stats["windows"] += 1

        stats["positive_windows"] += int(has_positive)

        stats["negative_only_windows"] += int(has_negative and not has_positive)

        stats["mixed_sign_windows"] += int(has_positive and has_negative)

        stats["mixed_regular_negative_windows"] += int(has_positive and regular_negative.any())

        stats["mixed_explicit_negative_windows"] += int(has_positive and explicit_negative.any())

        stats["supervised_occurrences"] += int(valid.sum())

        stats["positive_occurrences"] += int(positive.sum())

        stats["regular_negative_occurrences"] += int(regular_negative.sum())

        stats["explicit_negative_occurrences"] += int(explicit_negative.sum())



        y0, _, x0, _ = dataset._mt_center_bounds(y_off, x_off)

        for index in np.flatnonzero(valid):

            iy, ix = divmod(int(index), n)

            ys = y0 + iy * sub

            xs = x0 + ix * sub

            cy = (ys + sub // 2) // sub

            cx = (xs + sub // 2) // sub

            if not (0 <= cy < maps["positive"].shape[0] and 0 <= cx < maps["positive"].shape[1]):

                continue

            if labels[index] > 0:

                maps["positive"][cy, cx] = True

                maps["positive_hits"][cy, cx] += 1

            else:

                maps["negative"][cy, cx] = True

                maps["negative_hits"][cy, cx] += 1

                if labels[index] < 0:

                    maps["explicit_negative"][cy, cx] = True

    return stats





print("Mixed-window statistics now separate regular and explicit negatives")

In [ ]:
import importlib

import utils.dataloader as dataloader_module



dataloader_module = importlib.reload(dataloader_module)

DataManager = dataloader_module.DataManager



if "_base_ring_audit_config" not in globals():

    _base_ring_audit_config = _audit_config





def _audit_config(

    *,

    ring_negatives=True,

    ring_source="closed",

    close_r=3,

    gap_r=3,

    shell_r=2,

    pos_only=True,

    label_dilate_r=0,

):

    cfg = _base_ring_audit_config(

        ring_negatives=ring_negatives,

        ring_source=ring_source,

        close_r=close_r,

        gap_r=gap_r,

        shell_r=shell_r,

        pos_only=pos_only,

    )

    cfg.data.label_dilate_r = int(label_dilate_r)

    return cfg





print("Ring audit accepts controlled label dilation")

In [ ]:
OVERLAY_ALPHA = 0.5





def _blend_overlay(rgb, mask, color, alpha=OVERLAY_ALPHA):

    if not np.any(mask):

        return

    base = rgb[mask].astype(np.float32)

    overlay = np.asarray(color, dtype=np.float32)

    rgb[mask] = np.clip((1.0 - alpha) * base + alpha * overlay, 0, 255).astype(np.uint8)





def audit_ring_scenario(scroll_id, name, scenario, **options):

    cfg = _audit_config(**options)

    scroll_index = list(_SCROLL_IDS).index(int(scroll_id))

    dm = DataManager(

        cfg,

        scroll_id=int(scroll_id),

        domain_id=scroll_index,

        character_namespace=scroll_index,

    )

    train_set, valid_set = dm.get_datasets()

    height, width = dm.labels.shape

    sub = int(cfg.model.multitile_subtile)

    shape = (int(np.ceil(height / sub)) + 1, int(np.ceil(width / sub)) + 1)

    train_maps = _empty_audit_maps(shape)

    valid_maps = _empty_audit_maps(shape)

    train_stats = _scenario_stats(train_maps, _collect_scenario_targets(train_set, train_maps))

    valid_stats = _scenario_stats(valid_maps, _collect_scenario_targets(valid_set, valid_maps))



    raw, raw_source = _real_inklabel(scroll_id)

    background = cv2.resize(raw, (shape[1], shape[0]), interpolation=cv2.INTER_AREA)

    rgb = np.repeat(background[..., None], 3, axis=2).astype(np.uint8)

    for mask, color in (

        (train_maps["negative"], COLORS["train_papyrus"]),

        (valid_maps["negative"], COLORS["valid_papyrus"]),

        (train_maps["positive"], COLORS["train_ink"]),

        (valid_maps["positive"], COLORS["valid_ink"]),

        (train_maps["explicit_negative"], COLORS["explicit_negative"]),

    ):

        _blend_overlay(rgb, mask, color)



    fig, ax = plt.subplots(figsize=(18, max(6, 18 * height / width)))

    ax.imshow(rgb, interpolation="nearest")

    ax.set_title(

        f"{name} — {scroll_id} — {scenario}\n"

        f"50% supervision overlay; ring={cfg.data.ring_negatives}, "

        f"source={cfg.data.ring_label_source}, dilate={cfg.data.label_dilate_r}px, "

        f"close={cfg.data.ring_close_r}, gap={cfg.data.ring_gap_r}, "

        f"shell={cfg.data.ring_shell_r}, pos_only={cfg.data.multitile_pos_only}"

    )

    ax.axis("off")

    ax.legend(handles=[

        mpatches.Patch(color=COLORS["train_ink"] / 255, alpha=OVERLAY_ALPHA, label=f"train ink ({train_stats['unique_positive_cells']:,})"),

        mpatches.Patch(color=COLORS["train_papyrus"] / 255, alpha=OVERLAY_ALPHA, label=f"train negative ({train_stats['unique_negative_cells']:,})"),

        mpatches.Patch(color=COLORS["valid_ink"] / 255, alpha=OVERLAY_ALPHA, label=f"validation ink ({valid_stats['unique_positive_cells']:,})"),

        mpatches.Patch(color=COLORS["valid_papyrus"] / 255, alpha=OVERLAY_ALPHA, label=f"validation negative ({valid_stats['unique_negative_cells']:,})"),

        mpatches.Patch(color=COLORS["explicit_negative"] / 255, alpha=OVERLAY_ALPHA, label=f"explicit negative ({train_stats['unique_explicit_negative_cells']:,})"),

        mpatches.Patch(color=(0.5, 0.5, 0.5), label=f"ink reference ({raw_source})"),

    ], loc="upper right", fontsize=9, framealpha=0.95)

    plt.tight_layout()



    if SAVE_PNG:

        safe_name = "".join(ch if ch.isalnum() or ch in "-_" else "_" for ch in scenario)

        out_path = OUT_DIR / f"{scroll_id}_{safe_name}.png"

        fig.savefig(out_path, dpi=180, bbox_inches="tight")

        print(f"saved {out_path}")

    print(f"{scenario} train: {train_stats}")

    print(f"{scenario} valid: {valid_stats}")

    plt.show()



    del train_set, valid_set, dm, raw, background, rgb, train_maps, valid_maps

    gc.collect()

    return {"scenario": scenario, "train": train_stats, "valid": valid_stats}





print("Ring audit overlays use 50% alpha")

In [ ]:
W044_SCENARIOS = [

    (

        "current_closed_c3_g3_s2_pos_only",

        dict(

            ring_negatives=True,

            ring_source="closed",

            close_r=3,

            gap_r=3,

            shell_r=2,

            pos_only=True,

            label_dilate_r=0,

        ),

    ),

    (

        "campaign26_uneroded8_gap1_shell2_all_cells",

        dict(

            ring_negatives=True,

            ring_source="closed",

            close_r=0,

            gap_r=1,

            shell_r=2,

            pos_only=False,

            label_dilate_r=8,

        ),

    ),

    (

        "campaign26_eroded_touching_shell1_all_cells",

        dict(

            ring_negatives=True,

            ring_source="eroded",

            close_r=0,

            gap_r=0,

            shell_r=1,

            pos_only=False,

            label_dilate_r=0,

        ),

    ),

]



w044_results = []

for scenario_name, scenario_options in W044_SCENARIOS:

    w044_results.append(

        audit_ring_scenario(

            20260115000000,

            SCROLL_NAMES[20260115000000],

            scenario_name,

            **scenario_options,

        )

    )



print("\nw044 comparison")

for result in w044_results:

    train = result["train"]

    valid = result["valid"]

    print(

        f"{result['scenario']}: "

        f"train cells +{train['unique_positive_cells']:,}/-{train['unique_negative_cells']:,}, "

        f"both={train['cells_with_both_labels']:,}, "

        f"mixed regular={train['mixed_regular_negative_windows']:,}, "

        f"boundary edges={train['positive_negative_boundary_edges']:,}; "

        f"valid cells +{valid['unique_positive_cells']:,}/-{valid['unique_negative_cells']:,}, "

        f"both={valid['cells_with_both_labels']:,}, "

        f"mixed regular={valid['mixed_regular_negative_windows']:,}, "

        f"boundary edges={valid['positive_negative_boundary_edges']:,}"

    )